#Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/CS263

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/CS263


In [ ]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

In [ ]:
#import dataset
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/CS263/CS263_dataset.csv')
df.head()

,id,category,interaction,culture,label,confidence_score,cultural_context
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",acceptable,0.86,Adult autonomy and privacy are commonly respec...
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",context-dependent,0.78,Major life decisions are often shared with fam...
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",acceptable,0.84,Personal time and emotional boundaries are gen...
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm",context-dependent,0.76,"Family gatherings can be socially important, b..."
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",context-dependent,0.80,Individual financial responsibility is emphasi...


#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [ ]:
import re
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score

# 1. Load dataset
df = pd.read_csv('CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token
)

candidate_labels = ["acceptable behavior", "unacceptable behavior"]

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

# 5. Predict
predictions = []

for _, row in df.iterrows():

    pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)

results_df.head()
results_df.to_csv("deberta_predictions.csv", index=False)

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

Error during conversion: AttributeError("'str' object has no attribute 'decode'")
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
!pip install openai

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

#ChatGPT

In [ ]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Do not use cultural context unless it is explicitly included in the situation.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions.csv", index=False)

In [ ]:
SYSTEM_PROMPT = """
You are a careful social acceptability classifier.

Classify the interaction into exactly one label:
- acceptable
- not acceptable
- context-dependent

You are given:
1. interaction: the user/LLM exchange
2. culture: the relevant cultural or social norm
3. cultural_context: explanation of that norm

Use the provided culture and cultural_context when judging acceptability.
Do not invent additional cultural assumptions beyond what is provided.

Label definitions:
- acceptable: The behavior is generally socially appropriate under the given cultural context.
- not acceptable: The behavior clearly violates the given cultural/social norm, or is rude, harmful, invasive, disrespectful, unsafe, or unfair.
- context-dependent: The acceptability depends on missing details such as tone, closeness, consent, setting, urgency, frequency, age, relationship, or power dynamics.

Decision rules:
1. If privacy, consent, harassment, dishonesty, safety, or discrimination is involved, be stricter.
2. If the provided cultural context says expectations vary or depend on circumstances, prefer context-dependent.
3. If the behavior is a respectful boundary and the context supports autonomy/privacy, prefer acceptable.
4. If the behavior publicly embarrasses, invades privacy, violates trust, or disrespects a strong norm, prefer not acceptable.
5. Think internally before deciding, but output only valid JSON.

Return exactly this JSON format:
{"label": "acceptable"}
"""

def updated_predict_acceptability_gpt(row):
    user_prompt = f"""
Interaction:
{row["interaction"]}

Culture:
{row["culture"]}

Cultural context:
{row["cultural_context"]}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
from tqdm import tqdm

In [ ]:
# -------------------------
# 4. Run predictions
# -------------------------
gpt_predictions = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_label = updated_predict_acceptability_gpt(row)

    gpt_predictions.append({
        "id": row["id"],
        "category": row["category"],
        "interaction": row["interaction"],
        "culture": row["culture"],
        "cultural_context": row["cultural_context"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()

100%|██████████| 120/120 [02:04<00:00,  1.04s/it]


,id,category,interaction,culture,cultural_context,gold_label,prediction_label
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",Adult autonomy and privacy are commonly respec...,acceptable,acceptable
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",Major life decisions are often shared with fam...,context-dependent,context-dependent
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",Personal time and emotional boundaries are gen...,acceptable,acceptable
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm","Family gatherings can be socially important, b...",context-dependent,context-dependent
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",Individual financial responsibility is emphasi...,context-dependent,context-dependent


In [ ]:
# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")

GPT Accuracy: 0.8750


In [ ]:
FEW_SHOT = """
Examples:

Interaction:
User: I looked through my partner’s phone because I felt suspicious.
LLM: That raises privacy and trust concerns.
Culture:
US, relationship-privacy norm
Cultural context:
Privacy and trust are important in relationships.
Output:
{"label": "not acceptable"}

Interaction:
User: I skipped a family dinner because I wanted alone time.
LLM: That’s understandable. Did you explain it?
Culture:
US, personal-boundary norm
Cultural context:
Personal time and emotional boundaries are generally accepted, especially if communicated respectfully.
Output:
{"label": "acceptable"}

Interaction:
User: I didn’t help my sibling financially.
LLM: That depends on your situation.
Culture:
India, extended-family norm
Cultural context:
Family support may be expected, but obligation depends on resources, relationship, and circumstances.
Output:
{"label": "context-dependent"}
"""

SYSTEM_PROMPT = SYSTEM_PROMPT + "\n\n" + FEW_SHOT

In [ ]:
# -------------------------
# 4. Run predictions
# -------------------------
gpt_predictions = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred_label = updated_predict_acceptability_gpt(row)

    gpt_predictions.append({
        "id": row["id"],
        "category": row["category"],
        "interaction": row["interaction"],
        "culture": row["culture"],
        "cultural_context": row["cultural_context"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()

100%|██████████| 120/120 [01:28<00:00,  1.35it/s]


,id,category,interaction,culture,cultural_context,gold_label,prediction_label
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",Adult autonomy and privacy are commonly respec...,acceptable,acceptable
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",Major life decisions are often shared with fam...,context-dependent,context-dependent
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",Personal time and emotional boundaries are gen...,acceptable,acceptable
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm","Family gatherings can be socially important, b...",context-dependent,context-dependent
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",Individual financial responsibility is emphasi...,context-dependent,context-dependent


In [ ]:
# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"GPT Accuracy: {gpt_acc:.4f}")

GPT Accuracy: 0.8917
